# 股票预测流程走读（PIPELINE_WALKTHROUGH）

这个 notebook 把本仓库的股票预测流程摊平成可执行的 cell，逐步展示 **训练前 / 训练中 / 训练后** 各发生了什么。所有代码只读调用 `stock/` 下的现有模块，不修改任何模块代码；中间产物全部落在 `/tmp/ptw_run/`。

**整体流程（标注每步产物）**

```text
DuckDB  ../AutoTS/download/autots.duckdb（只读，绝不写入）
  │
  ├─[1] 股票池粗筛   stock.selection.universe
  │        → universe.csv / universe_meta.json（筛选漏斗与剔除原因）
  ├─[2] 单股导出     stock.data.export（内部 dataset.prepare_one）
  │        build_features → fit_preprocessing（只 fit train 窗口）→ make_samples
  │        → feature_matrix.npy / dates.npy / samples.csv / preprocessing.json / latest.npz ...
  ├─[3] 面板装载     stock.data.panel.load_catalog + PanelWindows
  │        → batch: (x, y, sid, target, y_raw, ex_mask, x_mark, head_cal)
  ├─[4] 模型         stock.adapters.timexer + models.TimeXer
  │        → forward: [B, 60, enc_in] → [B, 1, 1]
  ├─[5] 训练         stock.training.runner --mode train
  │        → checkpoint.pth / history.json / selection_result.json
  ├─[6] 评估         → test_predictions.csv / test_metrics.json
  │        + stock.evaluation.daily_hit（逐日选股胜率 / 基线 / oracle 上限）
  ├─[7] 选股         stock.selection.rules.hybrid_signal → top-N
  │        → stock.production.prediction_store（SQLite，插入即冻结）
  └─[8] 回测         stock.evaluation.backtest → 收盘→收盘 vs 可执行口径
```

**规模控制**：市值前 30 粗筛后取 6 只演示（其中 1 只预期在导出复检被剔除，实际 5 只进训练），`seq_len=60`、`pred_len=1`、`epochs=2`、`batch_size=64`、`d_model=32`。每一步打印真实数字。

| 阶段 | 输入 | 输出 | 关键文件 |
|---|---|---|---|
| 训练前 | DuckDB | 股票池、逐股特征/标准化/样本 | universe.csv, feature_matrix.npy, preprocessing.json, samples.csv |
| 训练中 | 导出目录 | 模型权重与训练记录 | checkpoint.pth, history.json, selection_result.json |
| 训练后 | checkpoint + test 段 | 指标、选股信号、回测报告 | test_metrics.json, daily_hit, predictions.db, backtest_report |


In [1]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

# 定位仓库根：notebook 位于 stock_timexer/，其父目录是仓库根
REPO = Path.cwd()
if not (REPO / "dataset.py").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

DB = REPO.parent / "AutoTS" / "download" / "autots.duckdb"   # 只读
RUN = Path("/tmp/ptw_run")      # 本 notebook 全部中间产物
shutil.rmtree(RUN, ignore_errors=True)
RUN.mkdir(parents=True)

def run_cli(*argv, timeout=3600):
    cmd = [sys.executable, "-m", *argv]
    print("$", " ".join(cmd))
    t0 = time.time()
    r = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True, timeout=timeout)
    print(f"[exit={r.returncode}, 耗时 {time.time()-t0:.1f}s]")
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"命令失败: {argv[0]}")
    return r

import torch, duckdb
print("repo    :", REPO)
print("python  :", sys.executable)
print("versions: torch", torch.__version__, "| duckdb", duckdb.__version__, "| pandas", pd.__version__)
print("db      :", DB, "| exists:", DB.exists())
print("HEAD    :", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True, cwd=str(REPO)).stdout.strip())


repo    : /root/projects/Time-Series-Library
python  : /root/projects/Time-Series-Library/.venv/bin/python
versions: torch 2.7.1+cu126 | duckdb 1.5.6 | pandas 3.0.6
db      : /root/projects/AutoTS/download/autots.duckdb | exists: True
HEAD    : e8ec8c8 P7: PanelWindows 矩阵缓存上限 64→512（1900 股微基准：0.164→0.130ms/读取，fd 仍低于 ulimit 1024）


## 2. 读库：DuckDB 里有什么

数据库是 AutoTS 采集库（**只读**）。各表在本流程中的角色：

| 表 | 内容 | 角色 |
|---|---|---|
| `akshare_tx` | 腾讯后复权(hfq)/不复权日线 OHLCV | 行情特征与标签来源；交易日历校准 |
| `stock_list` | 当前在市股票快照（简称/行业/市值/上市日） | 股票池排名、百度关键词映射（当前快照，非 PIT） |
| `baidu` | 百度指数（keyword × date × source） | 关注度外生特征 |
| `guzhi` | 估值 PE/PB/PS/PCF | 估值外生特征 |
| `rzrq` | 两融余额占比 | 杠杆资金外生特征 |
| `holiday_calendar` | 节假日 | 推导交易日历 |
| `tradingview_macro` / `tradingview_macro_symbols` | 宏观/指数行情 | 宏观外生特征 |
| `nbjb` / `stock_features` / `tradingview_study` | 其它采集数据 | 本流程不消费 |

下面以只读方式连接，列出各表行数与最新行情日。


In [2]:
con = duckdb.connect(str(DB), read_only=True)
tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
rows = {t: con.execute(f'SELECT count(*) FROM "{t}"').fetchone()[0] for t in tables}
overview = pd.DataFrame({"table": tables, "rows": [rows[t] for t in tables]})
latest = con.execute("SELECT max(date) FROM akshare_tx WHERE adjust='hfq'").fetchone()[0]
n_dates = con.execute("SELECT count(DISTINCT date) FROM akshare_tx WHERE adjust='hfq'").fetchone()[0]
print(f"最新 hfq 行情日: {latest} | hfq 覆盖交易日数: {n_dates}")
overview


最新 hfq 行情日: 2026-10-14 | hfq 覆盖交易日数: 8808


,table,rows
0,akshare_tx,15920867
1,baidu,26058514
2,guzhi,54896120
3,holiday_calendar,5113
4,nbjb,258631
5,rzrq,5730726
6,stock_features,15915832
7,stock_list,5224
8,tradingview_macro,83750
9,tradingview_macro_symbols,17


## 3. 股票池粗筛：stock.selection.universe

入选条件（缺省 = 最新时点）：① 市值排名前 `--top-mv`；② 后复权行情跨度 ≥ 3 年 **且** 有行情的不同交易日 ≥ 244×3；③ 两融表有记录；④ 百度指数有记录且最新观测距 as-of ≤ 30 天。漏斗逐阶段人数写入 `universe_meta.json`，**每阶段被剔除的数量就是相邻 stages 之差**。

坑：
- 市值/简称/行业是 **当前快照**（库内没有历史市值表），`--as-of` 只能做到"部分 PIT"；
- 粗筛只保证"库里有"，不保证百度/两融覆盖训练期 —— 最终资格由导出时的训练期覆盖率复检决定（见第 4 节）。

为控制规模只取市值前 30，再挑 6 只演示：**故意包含工商银行 601398**（两融记录很少），用来现场演示导出复检的剔除路径。


In [3]:
run_cli("stock.selection.universe", "--db", str(DB),
        "--out", str(RUN / "universe.csv"),
        "--top-mv", "30", "--min-years", "3", "--baidu-fresh-days", "30")

meta = json.loads((RUN / "universe_meta.json").read_text(encoding="utf-8"))
print("as_of =", meta["as_of"], "| 入选 =", meta["selected"])
stages = meta["stages"]
prev = None
print("\n筛选漏斗（阶段: 剩余数量 | 本阶段剔除）:")
for k, v in stages.items():
    drop = "" if prev is None else f"  ← 剔除 {prev - v}"
    print(f"  {k:<22} {v:>3}{drop}")
    prev = v

PICK = ["601398", "601939", "601288", "601988", "601857", "600519"]
uni = pd.read_csv(RUN / "universe.csv", dtype={"symbol": str})
pick = uni[uni["symbol"].isin(PICK)].sort_values("mv_rank")
pick.to_csv(RUN / "universe_pick.csv", index=False, encoding="utf-8-sig")
print("\n演示股票池（注意 601398 的 rzrq_rows 明显偏少）:")
pick[["symbol", "name", "industry_l2", "mv_rank", "quote_days", "baidu_days", "rzrq_rows"]]


$ /root/projects/Time-Series-Library/.venv/bin/python -m stock.selection.universe --db /root/projects/AutoTS/download/autots.duckdb --out /tmp/ptw_run/universe.csv --top-mv 30 --min-years 3 --baidu-fresh-days 30


[exit=0, 耗时 2.7s]
as_of=2026-10-14 | 筛选漏斗：{"top_mv": 30, "has_quotes": 30, "span_ge_1095d": 29, "quote_days_ge_732": 29, "has_rzrq": 29, "has_fresh_baidu": 29}
[OK] 入选 29 只股票 -> /tmp/ptw_run/universe.csv

as_of = 2026-10-14 | 入选 = 29

筛选漏斗（阶段: 剩余数量 | 本阶段剔除）:
  top_mv                  30
  has_quotes              30  ← 剔除 0
  span_ge_1095d           29  ← 剔除 1
  quote_days_ge_732       29  ← 剔除 0
  has_rzrq                29  ← 剔除 0
  has_fresh_baidu         29  ← 剔除 0

演示股票池（注意 601398 的 rzrq_rows 明显偏少）:


,symbol,name,industry_l2,mv_rank,quote_days,baidu_days,rzrq_rows
0,601398,工商银行,银行Ⅱ,2,4828,5765,510
1,601939,建设银行,银行Ⅱ,3,4611,5765,4015
2,601288,农业银行,银行Ⅱ,4,3940,5765,3934
3,601988,中国银行,银行Ⅱ,5,4911,5743,4013
5,601857,中国石油,炼化及贸易,7,4593,5743,4008
7,600519,贵州茅台,白酒Ⅱ,9,6020,5765,4012


## 4. 单股导出：stock.data.export

对每只股票调用 `dataset.prepare_one`：加载行情 → 按**可用时间假设**对齐辅助数据（观测日 + 发布滞后，禁止 bfill）→ `build_features` 构造特征 → `fit_preprocessing` **只用 train 窗口**拟合标准化 → `make_samples` 开窗。切分边界自动倒推：最新行情日起，最后 30 个交易日 = test，再往前 60 个交易日 = val，其余 = train。

产物（每只股票一个目录）：`aligned_values.csv`（对齐原值）、`features_unscaled.csv`（标准化前特征）、`feature_matrix.npy`（[T, F] float32 标准化矩阵）、`dates.npy`、`samples.csv`（窗口索引+切分+标签）、`train_preview.csv`、`latest.npz`（最新推理窗口）、`preprocessing.json`（冻结的标准化参数）、`manifest.json`（质量审计）。

**复检**：导出后检查 `feature_columns` 是否仍保留 `baidu_*` 与 `rzrq_*`（训练期覆盖率 ≥ 0.6 才会被保留），不满足者标记 `filtered_no_baidu` / `filtered_no_rzrq`，不进入共享训练。


In [4]:
run_cli("stock.data.export", "--db", str(DB),
        "--universe-csv", str(RUN / "universe_pick.csv"),
        "--data-root", str(RUN / "data"),
        "--workers", "6", "--seq-len", "60",
        "--test-days", "30", "--val-days", "60")

res = pd.read_csv(RUN / "data/export_results.csv", dtype={"symbol": str})
print("复检结果（只有 ok 进入训练）:")
print(res.to_string(index=False))
cfg = json.loads((RUN / "data/export_config.json").read_text(encoding="utf-8"))
print("\n切分边界: as_of =", cfg["as_of"], "| train_end =", cfg["train_end"],
      "| val_end =", cfg["val_end"], "| test_start =", cfg["test_start"])


$ /root/projects/Time-Series-Library/.venv/bin/python -m stock.data.export --db /root/projects/AutoTS/download/autots.duckdb --universe-csv /tmp/ptw_run/universe_pick.csv --data-root /tmp/ptw_run/data --workers 6 --seq-len 60 --test-days 30 --val-days 60


[exit=0, 耗时 11.5s]
{
  "export_version": "global_export_wf_v2",
  "split_layout": "train_val_test",
  "calendar_features": [
    "target_mark_dow",
    "target_mark_dom",
    "target_mark_doy",
    "target_gap_days_log",
    "target_is_gap_ge4",
    "target_is_pre_long_holiday",
    "target_is_month_start_3",
    "target_is_month_end_3"
  ],
  "db": "/root/projects/AutoTS/download/autots.duckdb",
  "as_of": "2026-10-14",
  "train_end": "2026-06-01",
  "val_end": "2026-08-25",
  "test_start": "2026-08-26",
  "train_start": null,
  "seq_len": 60,
  "min_train_samples": 250,
  "min_eval_samples": 20,
  "symbols": [
    "600519",
    "601288",
    "601398",
    "601857",
    "601939",
    "601988"
  ],
  "dataset_version": "stock_timexer_v8",
  "calendar_start": "2016-01-01",
  "min_market_symbols": 50,
  "policies": {
    "baidu": {
      "lag_days": 2,
      "release_hour": 0,
      "max_age_days": 7
    },
    "guzhi": {
      "lag_days": 2,
      "release_hour": 0,
      "max_age_days"

In [5]:
SYM = "600519"   # 以贵州茅台为例检查产物
d = RUN / "data" / SYM
print("产物清单:", sorted(p.name for p in d.iterdir()))

mat = np.load(d / "feature_matrix.npy")
dates = np.load(d / "dates.npy")
samples = pd.read_csv(d / "samples.csv")
prep = json.loads((d / "preprocessing.json").read_text(encoding="utf-8"))
print(f"\nfeature_matrix.npy: shape={mat.shape} dtype={mat.dtype}（T=交易日, F=保留特征列）")
print(f"dates.npy: shape={dates.shape}, {dates[0]} → {dates[-1]}")
print(f"preprocessing.feature_columns: {len(prep['feature_columns'])} 列，最后一列是目标列 {prep['feature_columns'][-1]!r}")
print(f"samples.csv: {samples.shape[0]} 行 × {samples.shape[1]} 列，列={list(samples.columns)}")
samples.head(3)


产物清单: ['aligned_values.csv', 'alignment_audit.csv', 'dates.npy', 'feature_matrix.npy', 'feature_schema.json', 'features_unscaled.csv', 'latest.npz', 'manifest.json', 'preprocessing.json', 'samples.csv', 'train_preview.csv']

feature_matrix.npy: shape=(2616, 146) dtype=float32（T=交易日, F=保留特征列）
dates.npy: shape=(2616,), 2016-01-04 → 2026-10-14
preprocessing.feature_columns: 146 列，最后一列是目标列 'stock_return'
samples.csv: 2555 行 × 9 列，列=['start_pos', 'end_pos', 'target_pos', 'input_date', 'target_date', 'split', 'label_valid', 'y_raw', 'y_scaled']


,start_pos,end_pos,target_pos,input_date,target_date,split,label_valid,y_raw,y_scaled
0,1,60,61,2016-04-05,2016-04-06,train,True,0.043588,2.540547
1,2,61,62,2016-04-06,2016-04-07,train,True,-0.011078,-0.713590
2,3,62,63,2016-04-07,2016-04-08,train,True,-0.015356,-0.968228


## 5. 特征与目标：每一列是什么，标签怎么定义

列命名约定（单一事实源：`stock/data/schema.py`）：

- `px_*`：行情派生特征（`px_volume` 已 log1p；OHLC 四个原值列只用于派生，不进模型 schema）；
- `baidu_* / val_* / rzrq_* / macro_*`：辅助外生来源。每个来源列带两个伴随列：
  - `*__missing`：0/1 缺失标记（填补用中位数，标记本身是信息）；
  - `*__age_days`：该值观测日距决策日的天数（可用时间假设的显式化）；
- `target_*`：**目标日（下一交易日）日历列**。日历永远完全已知，恒不缺失、不标准化、不参与屏蔽；
- `stock_return`：目标列 `TARGET` = `close(t+1)/close(t) − 1`（简单收益率、后复权），固定在矩阵最后一列。

`label_valid`：目标日行情有效且标签有限 = 可评分。输入窗口含无效核心行（如停牌）的样本被剔除；输入窗口有效但目标日无标签的行保留为 `label_valid=False`（可预测、不可评分，只是覆盖登记）。


In [6]:
feat_head = pd.read_csv(d / "features_unscaled.csv", nrows=3, index_col="date")
cols = list(feat_head.columns)

def col_group(c):
    if c == "stock_return":
        return "目标"
    if c.startswith("target_"):
        return "目标日日历"
    if c.endswith("__missing"):
        return "缺失标记"
    if c.endswith("__age_days"):
        return "数据年龄"
    for p, g in (("px_", "行情"), ("baidu_", "百度"), ("val_", "估值"),
                 ("rzrq_", "两融"), ("macro_", "宏观")):
        if c.startswith(p):
            return g
    return "其它"

from collections import defaultdict
groups = defaultdict(list)
for c in cols:
    groups[col_group(c)].append(c)
print(f"features_unscaled.csv（标准化前特征）共 {len(cols)} 列：")
for g, items in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(f"  {g:<6} {len(items):>3} 列  例: {items[:4]}")


features_unscaled.csv（标准化前特征）共 167 列：
  缺失标记    58 列  例: ['px_open__missing', 'px_close__missing', 'px_high__missing', 'px_low__missing']
  数据年龄    42 列  例: ['baidu_search_all__age_days', 'baidu_feed__age_days', 'val_pe__age_days', 'val_pb__age_days']
  宏观      34 列  例: ['macro_close_ATW1_4feedf8a', 'macro_volume_ATW1_4feedf8a', 'macro_close_CBOE_VIX_548b8a6f', 'macro_volume_CBOE_VIX_548b8a6f']
  行情      16 列  例: ['px_open', 'px_close', 'px_high', 'px_low']
  目标日日历    8 列  例: ['target_mark_dow', 'target_mark_dom', 'target_mark_doy', 'target_gap_days_log']
  估值       4 列  例: ['val_pe', 'val_pb', 'val_ps', 'val_pcf']
  百度       2 列  例: ['baidu_search_all', 'baidu_feed']
  两融       2 列  例: ['rzrq_fin_tval_ratio', 'rzrq_loan_tval_ratio']
  目标       1 列  例: ['stock_return']


In [7]:
from stock.data.targets import TARGET_SPEC
print("TARGET 定义登记（targets.TARGET_SPEC）:")
print(json.dumps(TARGET_SPEC, ensure_ascii=False, indent=2))

av = pd.read_csv(d / "aligned_values.csv", usecols=["date", "px_close"],
                 parse_dates=["date"], index_col="date")
s = samples[samples["label_valid"]].iloc[120]
c0 = float(av.loc[pd.Timestamp(s.input_date), "px_close"])
c1 = float(av.loc[pd.Timestamp(s.target_date), "px_close"])
calc = c1 / c0 - 1.0
print(f"\n用 {s.input_date} → {s.target_date} 验证标签:")
print(f"  close({s.input_date}) = {c0:.4f} → close({s.target_date}) = {c1:.4f}")
print(f"  手算 close(t+1)/close(t)-1 = {calc:.10f}")
print(f"  samples.csv 中 y_raw       = {s.y_raw:.10f}   一致 = {np.isclose(calc, s.y_raw)}")
print(f"\n该股 label_valid=False 行数: {int((~samples['label_valid']).sum())}"
      "（输入窗口有效、目标日无标签，仅作覆盖登记，不进训练/评估）")


TARGET 定义登记（targets.TARGET_SPEC）:
{
  "name": "stock_return",
  "horizon": 1,
  "definition": "next_trading_session_close_to_close",
  "unit": "fraction",
  "price_basis": "adjusted_close",
  "target_session": "next_trading_session",
  "suspension": "excluded; missing/invalid target-day OHLC => label_valid=False",
  "decision_time": "next_trading_session 08:30",
  "executable_return": "not this label; executable backtest enters at target-day open"
}

用 2016-09-27 → 2016-09-28 验证标签:
  close(2016-09-27) = 1800.9900 → close(2016-09-28) = 1785.4500
  手算 close(t+1)/close(t)-1 = -0.0086285876
  samples.csv 中 y_raw       = -0.0086285876   一致 = True

该股 label_valid=False 行数: 0（输入窗口有效、目标日无标签，仅作覆盖登记，不进训练/评估）


## 6. 标准化（防泄漏重点）：fit 只看 train 窗口

`fit_preprocessing` 对每列计算 median/mean/scale，拟合行集合 = **所有可评分训练样本窗口 [start_pos, end_pos] 覆盖到的行的并集**；拟合结果连同 `fit_first_date / fit_last_date / fit_row_count` 冻结进 `preprocessing.json`，推理（infer）原样复用。

如果用包含 val/test 的全部行去 fit，标准化参数就携带了未来评估期的分布信息 —— 一种微妙但真实的泄漏。下面先用代码**重建 fit 掩码**证明"fit 只看 train 窗口行"，再演示"若全样本 fit"的数值差异（只演示，不采用）。


In [8]:
print("登记的 fit 范围:", prep["fit_first_date"], "→", prep["fit_last_date"],
      f"| fit_row_count={prep['fit_row_count']} | train_end={prep['train_end']}")

feat = pd.read_csv(d / "features_unscaled.csv", index_col="date", parse_dates=True)
train_s = samples[(samples["split"] == "train") & samples["label_valid"]]
diff = np.zeros(len(feat) + 1, dtype=np.int64)
np.add.at(diff, train_s["start_pos"].to_numpy(dtype=np.int64), 1)
np.add.at(diff, train_s["end_pos"].to_numpy(dtype=np.int64) + 1, -1)
fit_mask = np.cumsum(diff[:-1]) > 0
fit = feat.loc[fit_mask]
print("由 samples.csv train 窗口重建: 行数 =", int(fit_mask.sum()),
      "| 范围 =", str(fit.index.min().date()), "→", str(fit.index.max().date()))
assert int(fit_mask.sum()) == prep["fit_row_count"]
assert str(fit.index.min().date()) == prep["fit_first_date"]
assert str(fit.index.max().date()) == prep["fit_last_date"]
print("→ 重建结果与登记完全一致：fit 确实只看了 train 窗口内的行\n")

print("重算几列统计量，与 preprocessing.json 比对:")
for col in ["px_return_5", "baidu_search_all", "rzrq_fin_tval_ratio", "stock_return"]:
    v = fit[col].dropna()
    med = float(v.median())
    filled = fit[col].fillna(med)
    mine = (med, float(filled.mean()), float(filled.std(ddof=0)))
    reg = prep["statistics"][col]
    ok = all(np.isclose(mine[i], reg[k], atol=1e-6)
             for i, k in enumerate(("median", "mean", "scale")))
    print(f"  {col:<22} 重算 median/mean/scale = {[round(x, 6) for x in mine]}  与登记一致 = {ok}")


登记的 fit 范围: 2016-01-05 → 2026-05-29 | fit_row_count=2524 | train_end=2026-06-01
由 samples.csv train 窗口重建: 行数 = 2524 | 范围 = 2016-01-05 → 2026-05-29
→ 重建结果与登记完全一致：fit 确实只看了 train 窗口内的行

重算几列统计量，与 preprocessing.json 比对:
  px_return_5            重算 median/mean/scale = [0.002105, 0.004525, 0.037606]  与登记一致 = True
  baidu_search_all       重算 median/mean/scale = [7.811163, 7.912394, 0.599633]  与登记一致 = True
  rzrq_fin_tval_ratio    重算 median/mean/scale = [-0.002122, -0.001626, 0.0362]  与登记一致 = True
  stock_return           重算 median/mean/scale = [-0.000222, 0.000909, 0.016799]  与登记一致 = True


In [9]:
print("【仅演示】若错误地用全部行（含 val/test）fit，参数会偏移多少：")
demo = []
for col in ["px_return_5", "baidu_search_all", "stock_return"]:
    st = prep["statistics"][col]
    filled_all = feat[col].fillna(feat[col].median())
    mean_all = float(filled_all.mean())
    scale_all = float(filled_all.std(ddof=0))
    demo.append({"列": col,
                 "train-fit mean": round(st["mean"], 8), "全样本 mean": round(mean_all, 8),
                 "Δmean": round(mean_all - st["mean"], 8),
                 "train-fit scale": round(st["scale"], 8), "全样本 scale": round(scale_all, 8),
                 "Δscale": round(scale_all - st["scale"], 8)})
print(pd.DataFrame(demo).to_string(index=False))
print("\n差异虽小但非零：val/test 的分布会渗进标准化参数，让评估指标乐观偏估。")
print("正式流程（fit_preprocessing）永远只 fit train 窗口；infer 复用训练期冻结参数。")


【仅演示】若错误地用全部行（含 val/test）fit，参数会偏移多少：
               列  train-fit mean  全样本 mean     Δmean  train-fit scale  全样本 scale    Δscale
     px_return_5        0.004525  0.004387 -0.000139         0.037606   0.037200 -0.000407
baidu_search_all        7.912394  7.886725 -0.025669         0.599633   0.609871  0.010238
    stock_return        0.000909  0.000877 -0.000033         0.016799   0.016636 -0.000163

差异虽小但非零：val/test 的分布会渗进标准化参数，让评估指标乐观偏估。
正式流程（fit_preprocessing）永远只 fit train 窗口；infer 复用训练期冻结参数。


## 7. 开窗：samples.csv

`make_samples` 的每一行是一个预测样本：输入窗口 `[start_pos, end_pos]`（长度 = seq_len，窗口末日 = 决策日），目标日 = `end_pos + 1`。split 按**目标日**划分：≤ train_end → train，≤ val_end → val，其余 → test。

剔除统计三类：`insufficient_history`（历史不足，结构性预热）、`invalid_input_window`（输入窗口含无效核心行，多为停牌）、`invalid_target_label`（输入窗口有效但目标日无标签 = `label_valid=False` 行数）。

验证：窗口长度恒为 60；`target_pos = end_pos + 1`；样本不跨股票（每只股票独立目录，装载时只在股内切片）；样本不跨 split（三个 split 的目标日区间互不重叠）。


In [10]:
ok_len = bool(((samples["end_pos"] - samples["start_pos"] + 1) == 60).all())
ok_tgt = bool((samples["target_pos"] == samples["end_pos"] + 1).all())
print(f"窗口长度恒为 60: {ok_len} | target_pos = end_pos+1: {ok_tgt}")

stat = samples.groupby("split").agg(
    n=("y_raw", "size"),
    label_valid=("label_valid", "sum"),
    first_target=("target_date", "min"),
    last_target=("target_date", "max"),
)
print("\n每个 split 的样本数与目标日范围:")
print(stat.to_string())

tr_max = stat.loc["train", "last_target"]
va_min, va_max = stat.loc["val", "first_target"], stat.loc["val", "last_target"]
te_min = stat.loc["test", "first_target"]
assert tr_max <= prep["train_end"] < va_min
assert va_max <= prep["val_end"] < te_min
print(f"\n边界检查: train.last={tr_max} ≤ train_end={prep['train_end']} < val.first={va_min}")
print(f"          val.last={va_max} ≤ val_end={prep['val_end']} < test.first={te_min}")
print("窗口不跨股票（单股目录 + 股内切片），也不跨 split（目标日区间互不重叠）")


窗口长度恒为 60: True | target_pos = end_pos+1: True

每个 split 的样本数与目标日范围:
          n  label_valid first_target last_target
split                                            
test     30           30   2026-08-26  2026-10-14
train  2465         2465   2016-04-06  2026-06-01
val      60           60   2026-06-02  2026-08-25

边界检查: train.last=2026-06-01 ≤ train_end=2026-06-01 < val.first=2026-06-02
          val.last=2026-08-25 ≤ val_end=2026-08-25 < test.first=2026-08-26
窗口不跨股票（单股目录 + 股内切片），也不跨 split（目标日区间互不重叠）


## 8. batch 组装：PanelWindows + DataLoader

`load_catalog` 先复检各股导出目录的口径一致性（切分边界/seq_len/日历列等），并生成**全局 schema**（各股保留列的并集，日历列排在外生列末尾，TARGET 恒为最后一列）。`PanelWindows` 把全局样本索引映射到股内位置，返回 8 元组 `(x, y, sid, target, y_raw, ex_mask, x_mark, head_cal)`。

- 某股缺少的全局列：补 0 + `__missing=1`（`align_features`），绝不借其它股票的值；
- `stock_weights`：让每个 epoch 内**各股票权重总和相等** → 单样本权重 = 总样本数 / (股票数 × 该股样本数)，防止样本多的股票主导损失；
- `sid` 只用于损失加权与按股取反归一化参数，**不进模型**：模型只看特征窗口；把股票 id 喂给模型等于让它死记个股水平，破坏跨股泛化。


In [11]:
import torch
from torch.utils.data import DataLoader
from stock.data.panel import load_catalog, PanelWindows

records, schema = load_catalog(str(RUN / "data"))
print("进入训练的股票:", [r["symbol"] for r in records])
print(f"全局 schema: {len(schema)} 列（各股保留列并集），最后一列 = {schema[-1]!r}")

ds_tr = PanelWindows(records, schema, "train", calendar_mode="time_marks")
ds_va = PanelWindows(records, schema, "val", allow_missing=True, calendar_mode="time_marks")
ds_te = PanelWindows(records, schema, "test", allow_missing=True, calendar_mode="time_marks")
print(f"样本数 train/val/test: {len(ds_tr)} / {len(ds_va)} / {len(ds_te)}")

print("\nstock_weights 配平（每股 样本数×权重 = 总样本数/股票数）:")
per = len(ds_tr) / len(ds_tr.record_ids)
for lid, rid in enumerate(ds_tr.record_ids):
    w = float(ds_tr.stock_weights[rid])
    print(f"  {records[rid]['symbol']}: 样本={int(ds_tr.counts[lid]):>5}  权重={w:.6f}"
          f"  样本×权重={ds_tr.counts[lid] * w:.1f}（目标 {per:.1f}）")

loader = DataLoader(ds_tr, batch_size=64, shuffle=False, num_workers=0)
x, y, sid, target, y_raw, ex_mask, x_mark, head_cal = next(iter(loader))
print("\n取一个 batch（shuffle=False，前 64 个训练样本）:")
for name, t in (("x", x), ("y", y), ("sid", sid), ("target", target), ("y_raw", y_raw),
                ("ex_mask", ex_mask), ("x_mark", x_mark), ("head_cal", head_cal)):
    print(f"  {name:<9} shape={tuple(t.shape)}  dtype={t.dtype}")
print(f"\n本 batch ex_mask=True 计数: {int(ex_mask.sum())}"
      "（整窗无观测的外生 token；是否传给模型由 --ex-mask 决定）")
print(f"head_cal 首样本（目标日星期 one-hot，5 维）: {head_cal[0].tolist()}")


进入训练的股票: ['600519', '601288', '601857', '601939', '601988']
全局 schema: 138 列（各股保留列并集），最后一列 = 'stock_return'


样本数 train/val/test: 12325 / 300 / 150

stock_weights 配平（每股 样本数×权重 = 总样本数/股票数）:
  600519: 样本= 2465  权重=1.000000  样本×权重=2465.0（目标 2465.0）
  601288: 样本= 2465  权重=1.000000  样本×权重=2465.0（目标 2465.0）
  601857: 样本= 2465  权重=1.000000  样本×权重=2465.0（目标 2465.0）
  601939: 样本= 2465  权重=1.000000  样本×权重=2465.0（目标 2465.0）
  601988: 样本= 2465  权重=1.000000  样本×权重=2465.0（目标 2465.0）



取一个 batch（shuffle=False，前 64 个训练样本）:
  x         shape=(64, 60, 135)  dtype=torch.float32
  y         shape=(64, 1, 1)  dtype=torch.float32
  sid       shape=(64,)  dtype=torch.int64
  target    shape=(64,)  dtype=torch.int64
  y_raw     shape=(64,)  dtype=torch.float64
  ex_mask   shape=(64, 134)  dtype=torch.bool
  x_mark    shape=(64, 60, 3)  dtype=torch.float32
  head_cal  shape=(64, 5)  dtype=torch.float32

本 batch ex_mask=True 计数: 64（整窗无观测的外生 token；是否传给模型由 --ex-mask 决定）
head_cal 首样本（目标日星期 one-hot，5 维）: [0.0, 0.0, 1.0, 0.0, 0.0]


## 9. 模型前向：TimeXer 适配器

TimeXer 双分支：**内生分支**对本股特征做非重叠 patch（`patch_len` 必须整除 `seq_len`，否则最近的目标历史会被截断）；**外生分支**把每个外生变量嵌入为一个 token（inverted embedding），本仓库再叠加 token 身份编码（`TypedExogenousEmbedding`）。

`calendar_mode=time_marks`：3 个目标日时间标记列（year/month/day）移入 `x_mark` 通道，5 个 flag 列（星期/月末/季末/年末/节前）留在输入列；预测头再拼接目标日星期 one-hot（`head_calendar_dim=5`）。

token 构成：内生 = `enc_in` 个变量 × (patch 数 + 1 个全局 token)；外生 = (`enc_in − 1`) 个值 token + `mark_dim` 个时间标记 token。


In [12]:
from argparse import Namespace
from stock.adapters import get_adapter

adapter = get_adapter("TimeXer")
args = Namespace(d_model=32, e_layers=2, n_heads=4, dropout=0.1, patch_len=10,
                 ex_mask=False, calendar_mode="time_marks", head_calendar=True)
adapter.validate_args(args, 60)
config = adapter.model_config(args, 60, schema)
print("enc_in =", config["enc_in"], "| calendar_mark_dim =", config["calendar_mark_dim"],
      "| head_calendar_dim =", config["head_calendar_dim"], "| pred_len =", config["pred_len"])
print("日历列（flag 列留在输入列）:", config["calendar_columns"])

patch_num = config["seq_len"] // config["patch_len"]
en_tok = config["enc_in"] * (patch_num + 1)
ex_tok = (config["enc_in"] - 1) + config["calendar_mark_dim"]
print(f"\n内生分支: {config['seq_len']}/{config['patch_len']} = {patch_num} 个 patch + 1 个全局 token"
      f" → 每变量 {patch_num + 1} 个 token，共 {config['enc_in']}×{patch_num + 1} = {en_tok}")
print(f"外生分支: (enc_in−1) = {config['enc_in'] - 1} 个值 token + {config['calendar_mark_dim']} 个时间标记 token = {ex_tok}")
print(f"注意力总序列长度 = {en_tok} + {ex_tok} = {en_tok + ex_tok}")

model = adapter.build_model(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"\n参数量: {n_params:,}")
model.eval()
with torch.no_grad():
    out = adapter.forward(model, x, ex_mask=None,
                          x_mark=x_mark if x_mark.shape[-1] else None,
                          head_calendar=head_cal)
print(f"前向输出形状: {tuple(out.shape)}（B, pred_len=1, 1），未标准化空间的收益预测")


enc_in = 135 | calendar_mark_dim = 3 | head_calendar_dim = 5 | pred_len = 1
日历列（flag 列留在输入列）: ['target_mark_dow', 'target_mark_dom', 'target_mark_doy', 'target_gap_days_log', 'target_is_gap_ge4', 'target_is_pre_long_holiday', 'target_is_month_start_3', 'target_is_month_end_3']

内生分支: 60/10 = 6 个 patch + 1 个全局 token → 每变量 7 个 token，共 135×7 = 945
外生分支: (enc_in−1) = 134 个值 token + 3 个时间标记 token = 137
注意力总序列长度 = 945 + 137 = 1082

参数量: 32,646
前向输出形状: (64, 1, 1)（B, pred_len=1, 1），未标准化空间的收益预测


## 10. 训练：stock.training.runner --mode train

训练协议：AdamW + 梯度裁剪 1.0；逐样本损失乘 `stock_weights`（按股等权）；按 val loss 早停（`--patience`）；最优 checkpoint 连同**完整元数据**（schema / feature_schema_hash / dataset_id / calendar_mode / 各股 preprocessing 参数等）一起保存，推理时可完整复现。训练结束后立即在 test 段预测，落 `test_predictions.csv` / `test_metrics.json`。

这里只跑 2 个 epoch 演示全链路（生产建议 30 epochs + patience 5）。loss 为标准化空间的 MSE（按股等权聚合）。


In [13]:
run_cli("stock.training.runner", "--mode", "train",
        "--data-root", str(RUN / "data"), "--out-dir", str(RUN / "ckpt"),
        "--epochs", "2", "--patience", "2", "--batch-size", "64", "--workers", "0",
        "--d-model", "32", "--e-layers", "2", "--n-heads", "4", "--patch-len", "10",
        "--device", "cpu", "--seed", "42")


$ /root/projects/Time-Series-Library/.venv/bin/python -m stock.training.runner --mode train --data-root /tmp/ptw_run/data --out-dir /tmp/ptw_run/ckpt --epochs 2 --patience 2 --batch-size 64 --workers 0 --d-model 32 --e-layers 2 --n-heads 4 --patch-len 10 --device cpu --seed 42


[exit=0, 耗时 14.1s]
stocks=5, features=138, train=12325, val=300, device=cpu
[INFO] calendar_mode=time_marks calendar_columns=8 mark_dim=3 head_calendar_dim=5
epoch=1 train=1.074511 val=1.799196
epoch=2 train=0.983434 val=1.787485
{
  "y_pred": {
    "pooled": {
      "n": 150,
      "mae": 0.006997982086856385,
      "mse": 8.118733377530791e-05,
      "rmse": 0.009010401421430008,
      "direction_accuracy_nonzero": 0.5704697986577181
    },
    "stock_equal_mae": 0.006997982086856385,
    "stock_equal_mse": 8.118733377530791e-05,
    "mean_daily_rank_ic": null,
    "rank_ic_days": 0
  },
  "baseline_zero": {
    "pooled": {
      "n": 150,
      "mae": 0.006985062339941406,
      "mse": 8.157631249347725e-05,
      "rmse": 0.009031960611820518,
      "direction_accuracy_nonzero": 0.0
    },
    "stock_equal_mae": 0.006985062339941404,
    "stock_equal_mse": 8.157631249347724e-05,
    "mean_daily_rank_ic": null,
    "rank_ic_days": 0
  },
  "baseline_mean": {
    "pooled": {
      "n"

CompletedProcess(args=['/root/projects/Time-Series-Library/.venv/bin/python', '-m', 'stock.training.runner', '--mode', 'train', '--data-root', '/tmp/ptw_run/data', '--out-dir', '/tmp/ptw_run/ckpt', '--epochs', '2', '--patience', '2', '--batch-size', '64', '--workers', '0', '--d-model', '32', '--e-layers', '2', '--n-heads', '4', '--patch-len', '10', '--device', 'cpu', '--seed', '42'], returncode=0, stdout='stocks=5, features=138, train=12325, val=300, device=cpu\n[INFO] calendar_mode=time_marks calendar_columns=8 mark_dim=3 head_calendar_dim=5\nepoch=1 train=1.074511 val=1.799196\nepoch=2 train=0.983434 val=1.787485\n{\n  "y_pred": {\n    "pooled": {\n      "n": 150,\n      "mae": 0.006997982086856385,\n      "mse": 8.118733377530791e-05,\n      "rmse": 0.009010401421430008,\n      "direction_accuracy_nonzero": 0.5704697986577181\n    },\n    "stock_equal_mae": 0.006997982086856385,\n    "stock_equal_mse": 8.118733377530791e-05,\n    "mean_daily_rank_ic": null,\n    "rank_ic_days": 0\n 

In [14]:
ck = torch.load(RUN / "ckpt/checkpoint.pth", map_location="cpu", weights_only=True)
print("checkpoint 顶层键:", sorted(ck.keys()))
print("\nformat:", ck["format"], "| model:", ck["model"],
      "| best_epoch:", ck.get("best_epoch"),
      "| best_val_loss:", round(ck.get("best_val_loss"), 6))
mc = ck["model_config"]
print("model_config: enc_in =", mc["enc_in"], "| calendar_mode =", mc["calendar_mode"],
      "| ex_mask =", mc["ex_mask"], "| head_calendar_dim =", mc["head_calendar_dim"])
print("feature_schema_hash:", ck["feature_schema_hash"][:24], "...")
print("dataset_id:", ck["dataset_id"][:24], "...（= export_config.json 的 sha256，可追溯数据口径）")
print("schema 登记列数:", len(ck["schema"]),
      "| preprocessing 登记股票:", list(ck["preprocessing"].keys()))

hist = json.loads((RUN / "ckpt/history.json").read_text(encoding="utf-8"))
print("\nhistory.json:", json.dumps(hist, ensure_ascii=False))
sel = json.loads((RUN / "ckpt/selection_result.json").read_text(encoding="utf-8"))
print("selection_result: stopped_early =", sel["stopped_early"],
      "| completed_epochs =", sel["completed_epochs"],
      "| budget_rule =", sel["budget_rule"])


checkpoint 顶层键: ['best_epoch', 'best_val_loss', 'dataset_id', 'export_config', 'feature_schema', 'feature_schema_hash', 'format', 'model', 'model_config', 'preprocessing', 'schema', 'state_dict', 'target_spec', 'train_mean']

format: global_timexer_typed_v2 | model: TimeXer | best_epoch: 2 | best_val_loss: 1.787485
model_config: enc_in = 135 | calendar_mode = time_marks | ex_mask = False | head_calendar_dim = 5
feature_schema_hash: 7506b225dc0191cc94b8eab1 ...
dataset_id: 63b18637f4504c5e68fce2c2 ...（= export_config.json 的 sha256，可追溯数据口径）
schema 登记列数: 138 | preprocessing 登记股票: ['600519', '601288', '601857', '601939', '601988']

history.json: [{"epoch": 1, "train_stock_equal_scaled_mse": 1.0745114947477654, "val_stock_equal_scaled_mse": 1.7991959349314373}, {"epoch": 2, "train_stock_equal_scaled_mse": 0.9834343437732596, "val_stock_equal_scaled_mse": 1.7874847507476808}]
selection_result: stopped_early = False | completed_epochs = 2 | budget_rule = same_epochs


## 11. 评估：test 段指标 + 逐日胜率

`test_metrics.json`：pooled / stock_equal 的 MAE、MSE、方向准确率，并与三个基线对比（预测 0 / 预测 train 均值 / persistence 昨日收益）。注意：`n` 是**重叠窗口数**（相邻窗口共享 59 天），不是独立样本量；官方 `mean_daily_rank_ic` 要求每日 ≥ 10 只股票，5 只小池不会触发，下面直接手算日度 RankIC 作演示。

`daily_hit`：按日对 `y_pred` 排序取前 `ceil(n×q%)` 且预测收益 > floor 作为信号，命中 = 实际收益 > 0；同时给出逐日**基线上涨率**（随机选股的期望胜率）与 **oracle top-N 上限**（完美排序也选不出的收益，仅作天花板参考）。


In [15]:
tm = json.loads((RUN / "ckpt/test_metrics.json").read_text(encoding="utf-8"))
rows_m = []
for name in ("y_pred", "baseline_zero", "baseline_mean", "baseline_persistence"):
    p = tm[name]["pooled"]
    rows_m.append({"预测器": name, "n(重叠窗口)": p["n"],
                   "MAE": round(p["mae"], 6), "MSE": round(p["mse"], 8),
                   "方向准确率": None if p["direction_accuracy_nonzero"] is None
                                 else round(p["direction_accuracy_nonzero"], 4)})
print(pd.DataFrame(rows_m).to_string(index=False))

tp = pd.read_csv(RUN / "ckpt/test_predictions.csv", dtype={"symbol": str})
print(f"\ntest 预测: {len(tp)} 行 | {tp['symbol'].nunique()} 只股票 | {tp['target_date'].nunique()} 个交易日")

def day_ic(g):
    if len(g) < 3 or g["y_true"].nunique() < 2 or g["y_pred"].nunique() < 2:
        return np.nan
    return g["y_true"].rank().corr(g["y_pred"].rank())

ic = tp.groupby("target_date")[["y_true", "y_pred"]].apply(day_ic).dropna()
print(f"日度 RankIC（演示直算）: mean={ic.mean():.4f}  std={ic.std():.4f}"
      f"  min={ic.min():.4f}  max={ic.max():.4f}  天数={len(ic)}")
print("官方 mean_daily_rank_ic:", tm["y_pred"]["mean_daily_rank_ic"],
      "（要求每日 ≥10 只股票，小池不触发）")


                 预测器  n(重叠窗口)      MAE      MSE  方向准确率
              y_pred      150 0.006998 0.000081 0.5705
       baseline_zero      150 0.006985 0.000082 0.0000
       baseline_mean      150 0.006957 0.000081 0.5839
baseline_persistence      150 0.009852 0.000143 0.4899

test 预测: 150 行 | 5 只股票 | 30 个交易日
日度 RankIC（演示直算）: mean=-0.0733  std=0.5394  min=-1.0000  max=0.9000  天数=30
官方 mean_daily_rank_ic: None （要求每日 ≥10 只股票，小池不触发）


In [16]:
from stock.evaluation.daily_hit import daily_hit

for floor in (0.0, 0.5):
    daily, stats = daily_hit(tp, 20.0, floor)
    print(f"--- q=20%（5 只股票 → 每日 top-1），floor={floor}% ---")
    for k in ("days", "total_signals", "pooled_hit_rate", "avg_daily_hit_rate",
              "avg_daily_base_up_rate", "avg_oracle_hit_rate",
              "signal_mean_true_ret_pct", "all_mean_true_ret_pct"):
        v = stats[k]
        print(f"  {k:<28} {round(v, 4) if isinstance(v, float) else v}")
    print()

daily05, _ = daily_hit(tp, 20.0, 0.5)
print("逐日明细（floor=0.5%；base_up_rate=当日基线上涨率，oracle=完美排序上限）:")
print(daily05.round(3).to_string(index=False))


--- q=20%（5 只股票 → 每日 top-1），floor=0.0% ---
  days                         30
  total_signals                27
  pooled_hit_rate              0.4444
  avg_daily_hit_rate           0.4444
  avg_daily_base_up_rate       0.58
  avg_oracle_hit_rate          0.8667
  signal_mean_true_ret_pct     0.0448
  all_mean_true_ret_pct        0.1729



--- q=20%（5 只股票 → 每日 top-1），floor=0.5% ---
  days                         30
  total_signals                0
  pooled_hit_rate              None
  avg_daily_hit_rate           nan
  avg_daily_base_up_rate       0.58
  avg_oracle_hit_rate          0.8667
  signal_mean_true_ret_pct     None
  all_mean_true_ret_pct        0.1729



逐日明细（floor=0.5%；base_up_rate=当日基线上涨率，oracle=完美排序上限）:
target_date  signals  hits  hit_rate  day_up_base  day_mean_true_ret_pct  signal_mean_true_ret_pct  oracle_signals  oracle_hits  oracle_hit_rate
 2026-08-26        0     0       NaN          0.6                 -0.088                       NaN               1            1              1.0
 2026-08-27        0     0       NaN          0.2                 -0.430                       NaN               1            1              1.0
 2026-08-28        0     0       NaN          0.8                  0.500                       NaN               1            1              1.0
 2026-08-31        0     0       NaN          1.0                  1.566                       NaN               1            1              1.0
 2026-09-01        0     0       NaN          1.0                  0.751                       NaN               1            1              1.0
 2026-09-02        0     0       NaN          0.6                 -0.136     

## 12. 选股输出：hybrid_signal + SQLite 落库

生产日流水（`stock.production.daily_pipeline`）：最新窗口推理 → 全池打分 → `hybrid_signal`（当日预测前 q% **且** 预测收益 > floor）→ 落 SQLite。

`prediction_store` 两张表：`prediction_runs`（每次运行元数据）+ `predictions`（逐股预测），**唯一键 = PRIMARY KEY (run_id, symbol, target_date)**；BEFORE UPDATE/DELETE 触发器把记录冻结 —— 重跑只能新建 run_id，历史不可篡改。

下面用最后一个 test 日做选股（q=40% → top-2，模拟生产 top-N），并演示落到 `/tmp` 的临时 SQLite。


In [17]:
from stock.selection.rules import hybrid_signal
from stock.production import prediction_store as store
from stock.inference.contracts import normalize_prediction_frame
from stock.utils import sha256_file

last_day = tp["target_date"].max()
day = tp[tp["target_date"] == last_day].copy().reset_index(drop=True)
rule = hybrid_signal(day["y_pred"], 40.0, floor_decimal=0.005, pred_unit="decimal",
                     symbol=day["symbol"], eligible=day["symbol"].tolist())
day = pd.concat([day, rule], axis=1)
print(f"目标日 {last_day}（q=40% → top-2，floor=0.5%）:")
print(day[["symbol", "y_pred", "y_true", "rank", "quantile_pass", "floor_pass", "signal"]]
      .sort_values("rank").to_string(index=False))

# --- 落 SQLite 演示（/tmp 临时库） ---
smp = RUN / "data" / records[0]["symbol"] / "samples.csv"
input_dates = pd.read_csv(smp, usecols=["input_date", "target_date"], dtype=str)
as_of_date = input_dates.loc[input_dates["target_date"] == last_day, "input_date"].iloc[0]
ckpt_sha = sha256_file(str(RUN / "ckpt/checkpoint.pth"))

con_sql = store.connect(RUN / "predictions_demo.db")
run_id = store.make_run_id(con_sql, last_day, ckpt_sha)
store.register_run(con_sql, run_id=run_id, as_of_date=as_of_date, target_date=last_day,
                   model_id=ckpt_sha[:16], dataset_id=ck["dataset_id"],
                   checkpoint=str(RUN / "ckpt/checkpoint.pth"),
                   checkpoint_sha256=ckpt_sha,
                   feature_schema_hash=ck["feature_schema_hash"],
                   quantile_pct=40.0, floor_pct=0.5)
frame = normalize_prediction_frame(pd.DataFrame({
    "symbol": day["symbol"], "target_date": last_day, "as_of_date": as_of_date,
    "model_id": ckpt_sha[:16], "dataset_id": ck["dataset_id"],
    "y_pred": day["y_pred"], "status": "ok",
    "rank": day["rank"], "quantile_pass": day["quantile_pass"].astype(int),
    "floor_pass": day["floor_pass"].astype(int), "signal": day["signal"].astype(int),
}))
n = store.insert_predictions(con_sql, run_id, frame)
back = store.read_predictions(con_sql, run_id)
print(f"\n已写入 {n} 条 → {RUN / 'predictions_demo.db'}，读回 {len(back)} 行:")
print(back[["run_id", "symbol", "target_date", "pred_return_pct", "rank", "signal", "status"]]
      .to_string(index=False))

ddl = con_sql.execute("SELECT sql FROM sqlite_master WHERE name='predictions'").fetchone()[0]
print("\npredictions 表唯一键:", [l.strip() for l in ddl.splitlines() if "PRIMARY KEY" in l])
print("冻结触发器:", [r[0] for r in con_sql.execute(
    "SELECT name FROM sqlite_master WHERE type='trigger'").fetchall()])
try:
    con_sql.execute("UPDATE predictions SET y_pred=0 WHERE run_id=?", (run_id,))
except Exception as e:
    print(f"\n尝试篡改被拒绝: {type(e).__name__}: {e}")


目标日 2026-10-14（q=40% → top-2，floor=0.5%）:
symbol    y_pred    y_true  rank  quantile_pass  floor_pass  signal
600519  0.002600  0.006964   1.0           True       False   False
601288  0.001167 -0.006890   2.0           True       False   False
601939  0.000680  0.002257   3.0          False       False   False
601857  0.000331  0.017104   4.0          False       False   False
601988 -0.000126 -0.003518   5.0          False       False   False



已写入 5 条 → /tmp/ptw_run/predictions_demo.db，读回 5 行:
                               run_id symbol target_date  pred_return_pct  rank  signal status
run-20261014-20261103T204821-22bea4f3 600519  2026-10-14         0.260018     1       0     ok
run-20261014-20261103T204821-22bea4f3 601288  2026-10-14         0.116739     2       0     ok
run-20261014-20261103T204821-22bea4f3 601939  2026-10-14         0.067966     3       0     ok
run-20261014-20261103T204821-22bea4f3 601857  2026-10-14         0.033095     4       0     ok
run-20261014-20261103T204821-22bea4f3 601988  2026-10-14        -0.012624     5       0     ok

predictions 表唯一键: ['PRIMARY KEY (run_id, symbol, target_date),']
冻结触发器: ['predictions_frozen_update', 'predictions_frozen_delete']

尝试篡改被拒绝: IntegrityError: predictions 已冻结：重跑请新建 run_id，不要原地更新


## 13. 回测：收盘→收盘 vs 可执行口径

`daily_hit` 隐含"在目标日收盘价成交"（理论口径）；实际信号在 t+1 早上 08:30 才形成，最早只能按 **t+1 开盘价**买入。`run_backtest` 并列输出两种口径：

- `close_to_close`：直接用标签收益（与 daily_hit 同源）；
- `executable`：t+1 开盘买入、持有 `hold_days` 个交易日、收盘卖出，双边扣手续费 + 滑点。

从 DuckDB 取 5 只股票 test 段的 hfq 开盘/收盘作为成交价，双边各 10bp 费用 + 10bp 滑点。

注意：生产口径 floor=0.5% 在本小池子里会**零信号**（2 个 epoch 的模型，每日 top-1 预测收益到不了 0.5%），而且空信号会触发 `backtest.py` 的一个真实 bug（下面现场演示并记录）；因此正式回测用 floor=0（纯 top-q）跑通。


In [18]:
from stock.evaluation.backtest import run_backtest
from dataset import sql_symbol

val_end, as_of = cfg["val_end"], cfg["as_of"]
syms = sorted(tp["symbol"].unique())
ph = ",".join(["?"] * len(syms))
sql = (f"SELECT {sql_symbol('symbol')} AS symbol, date, open, close FROM akshare_tx "
       f"WHERE adjust='hfq' AND date > ? AND date <= ? "
       f"AND {sql_symbol('symbol')} IN ({ph})")
prices = con.execute(sql, [val_end, as_of, *syms]).fetchdf()
prices["date"] = pd.to_datetime(prices["date"]).dt.strftime("%Y-%m-%d")
print(f"成交价: {len(prices)} 行（{len(syms)} 只股票 × test 段交易日）")

# 先按生产口径 floor=0.5% 试：小池子 + 2 epoch 模型 → 零信号，并触发一个真实 bug
try:
    run_backtest(tp, prices=prices, q_pct=20.0, floor_pct=0.5,
                 hold_days=1, fee_bps=10.0, slippage_bps=10.0)
    print("floor=0.5%: 正常完成")
except KeyError as e:
    print(f"floor=0.5%: 零信号，并触发真实 bug —— KeyError {e}")
    print("  根因: signals 为空时 executable_returns 返回无列的 (0,0) 空表，")
    print("        trades.dropna(subset=['exec_return']) 直接抛错（应先判空或保持列模式）。")
    print("  已记录为 bug，不修改 stock/ 代码；下面用 floor=0 规避并跑通全链路。\n")

report, scored, signals = run_backtest(tp, prices=prices, q_pct=20.0, floor_pct=0.0,
                                       hold_days=1, fee_bps=10.0, slippage_bps=10.0)
print(f"floor=0（纯 top-q）: 信号 {report['signals']} 条 | 有信号天数 {report['signal_days']}/{report['days']}")
cc, ex = report["close_to_close"], report["executable"]
rows_bt = []
for name, b in (("收盘→收盘（理论）", cc), ("可执行（t+1 开盘，含费用）", ex)):
    rows_bt.append({
        "口径": name, "信号数": b["n"],
        "胜率": None if b["hit_rate"] is None else round(b["hit_rate"], 4),
        "日均收益%": None if b["daily_mean_return"] is None else round(b["daily_mean_return"] * 100, 4),
        "累计收益%": None if b["cumulative_return"] is None else round(b["cumulative_return"] * 100, 3),
        "最大回撤%": None if b["max_drawdown"] is None else round(b["max_drawdown"] * 100, 3),
    })
print(pd.DataFrame(rows_bt).to_string(index=False))
print("\n口径声明:", json.dumps(report["assumptions"], ensure_ascii=False, indent=1))
print(f"可执行口径无法成交的信号（缺价/停牌）: {ex['unpriced']} | 日均换手: {ex['avg_turnover']:.3f}")


成交价: 150 行（5 只股票 × test 段交易日）


floor=0.5%: 零信号，并触发真实 bug —— KeyError ['exec_return']
  根因: signals 为空时 executable_returns 返回无列的 (0,0) 空表，
        trades.dropna(subset=['exec_return']) 直接抛错（应先判空或保持列模式）。
  已记录为 bug，不修改 stock/ 代码；下面用 floor=0 规避并跑通全链路。



floor=0（纯 top-q）: 信号 27 条 | 有信号天数 27/30
             口径  信号数     胜率   日均收益%  累计收益%   最大回撤%
      收盘→收盘（理论）   27 0.4444  0.0448  1.080  -4.124
可执行（t+1 开盘，含费用）   27 0.2963 -0.3203 -8.401 -10.028

口径声明: {
 "signal": "每日 y_pred 前 20%（ceil）且 y_pred > 0%",
 "eligible": "当日有预测的全部股票（与 daily_hit 相同）",
 "close_to_close": "t 收盘 → t+1 收盘（后复权标签 y_true）",
 "executable": "t+1 开盘买入，持有 1 个交易日后收盘卖出；手续费 10bp/边，滑点 10bp/边",
 "weights": "信号股等权，无信号日空仓",
 "turnover": "0.5*(新增+剔除)/max(期初,期末持仓数)，日频再平衡近似"
}
可执行口径无法成交的信号（缺价/停牌）: 0 | 日均换手: 0.417


## 14. 总结：训练前 / 训练中 / 训练后

| 阶段 | 步骤 | 输入 | 输出 | 关键文件 | 易错点 |
|---|---|---|---|---|---|
| 训练前 | 股票池 | stock_list/baidu/rzrq/akshare_tx | universe.csv | universe_meta.json | 市值/简称是当前快照（部分 PIT）；粗筛不保证辅助数据训练期覆盖 |
| 训练前 | 导出 | universe + 行情 + 辅助数据 | 特征矩阵/标准化/样本 | feature_matrix.npy, preprocessing.json, samples.csv | fit 只用 train 窗口（防泄漏）；禁止 bfill；baidu/rzrq 复检决定能否进训练 |
| 训练前 | 开窗 | features + 切分边界 | samples.csv | samples.csv | label_valid=False 只是覆盖登记；窗口不跨股票、不跨 split |
| 训练中 | batch | 导出目录 | (x, y, sid, ...) batch | — | stock_weights 按股等权；sid 不进模型；缺列补 0 + __missing=1 |
| 训练中 | 训练 | PanelWindows | checkpoint.pth | history.json, selection_result.json | patch_len 必须整除 seq_len；早停看 val；calendar_mode/ex_mask 登记进 checkpoint |
| 训练后 | 评估 | checkpoint + test 段 | 指标 | test_predictions.csv, test_metrics.json | n 是重叠窗口数；官方日度 RankIC 需每日 ≥10 只 |
| 训练后 | 选股 | 预测值 | top-N 信号 | predictions.db（SQLite） | 唯一键 (run_id, symbol, target_date)；插入即冻结，重跑只能新 run |
| 训练后 | 回测 | 信号 + 成交价 | 两口径报告 | backtest_report | 收盘→收盘不可执行；可执行口径要 t+1 开盘 + 费用 |

**本走读的已知局限**：

- 2 epochs + 5 只股票只为跑通全链路，指标不代表生产水平（生产：全池数千只、30 epochs + 早停、walk-forward 滚动重训）；
- universe 的市值排名用当前快照，历史回测应使用 `--as-of`（部分 PIT）；
- 预测标签是收盘→收盘收益，信号在 t+1 08:30 形成，无法在 t 日收盘成交 —— 回测必须用可执行口径复核；
- 全部产物位于 `/tmp/ptw_run/`，重新运行本 notebook 会清空重建。

**过程中发现并记录的真实 bug（未修改 `stock/` 代码）**：

- `stock/evaluation/backtest.py::executable_returns`：当信号为空时返回无列的 `(0,0)` DataFrame，导致 `run_backtest` 在 `trades.dropna(subset=["exec_return"])` 抛 `KeyError`。本走读中由"5 只股票 + floor=0.5% → 零信号"触发；建议修复为空输入时返回带完整列模式的空表（或先判空）。
